# 3.0.1 Face AutoKeras Constrained Baseline

Purpose:
- Avoid RAM-heavy full-array loading
- Stream images using tf.data
- Apply training-only augmentation on the fly
- Use class weights instead of materialized augmented images
- Constrain AutoKeras to lightweight architecture search

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FACE_SPLIT_MANIFEST_PATH,
    FACE_IMAGE_SIZE,
    FACE_BATCH_SIZE,
    FACE_AUTOKERAS_TRIAL_001_DIR,
    FACE_AUTOKERAS_TRIAL_001_MODEL_PATH,
    FACE_AUTOKERAS_TRIAL_001_RESULTS_PATH,
    FACE_AUTOKERAS_TRIAL_001_PREDICTIONS_PATH,
    FACE_AUTOKERAS_TRIAL_001_CONFUSION_MATRIX_PATH,
)
from rural_stroke_assist.preprocessing.tf_image_dataset import (
    dataframe_to_tf_dataset,
    add_augmentation,
)
from rural_stroke_assist.preprocessing.face_dataset import (
    compute_class_weights_from_manifest,
    INVERSE_LABEL_MAPPING,
)
from rural_stroke_assist.modeling.autokeras_models import (
    build_constrained_image_automodel,
)
from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

c:\Users\Asus\Downloads\Uni_work\Grad-Project\autokeras_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tf.random.set_seed(42)

face_split_df = pd.read_csv(FACE_SPLIT_MANIFEST_PATH)

face_split_df.groupby(["split", "class_label"]).size()

split  class_label
test   NonStroke      208
       Stroke         110
train  NonStroke      970
       Stroke         510
val    NonStroke      208
       Stroke         109
dtype: int64

In [3]:
tf.random.set_seed(42)

face_split_df = pd.read_csv(FACE_SPLIT_MANIFEST_PATH)

face_split_df.groupby(["split", "class_label"]).size()

split  class_label
test   NonStroke      208
       Stroke         110
train  NonStroke      970
       Stroke         510
val    NonStroke      208
       Stroke         109
dtype: int64

In [4]:
train_df = face_split_df[face_split_df["split"] == "train"].copy()
val_df = face_split_df[face_split_df["split"] == "val"].copy()
test_df = face_split_df[face_split_df["split"] == "test"].copy()

len(train_df), len(val_df), len(test_df)

(1480, 317, 318)

In [5]:
class_weights = compute_class_weights_from_manifest(train_df)
class_weights

{0: 0.7628865979381443, 1: 1.4509803921568627}

In [6]:
train_ds = dataframe_to_tf_dataset(
    train_df,
    image_size=FACE_IMAGE_SIZE,
    batch_size=FACE_BATCH_SIZE,
    shuffle=True,
    seed=42,
)

train_ds = add_augmentation(train_ds)

val_ds = dataframe_to_tf_dataset(
    val_df,
    image_size=FACE_IMAGE_SIZE,
    batch_size=FACE_BATCH_SIZE,
    shuffle=False,
)

test_ds = dataframe_to_tf_dataset(
    test_df,
    image_size=FACE_IMAGE_SIZE,
    batch_size=FACE_BATCH_SIZE,
    shuffle=False,
)

In [7]:
for images, labels in train_ds.take(1):
    print(images.shape)
    print(labels.shape)
    print(tf.reduce_min(images).numpy(), tf.reduce_max(images).numpy())

(16, 160, 160, 3)
(16,)
0.0 1.0546887


In [9]:
automodel = build_constrained_image_automodel(
    block_type="vanilla",
    max_trials=2,
    project_name="face_autokeras_vanilla_160",
    directory=str(FACE_AUTOKERAS_TRIAL_001_DIR),
    overwrite=True,
)

In [10]:
history = automodel.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weights,
)

Trial 2 Complete [00h 02m 13s]
val_loss: 0.6913703083992004

Best val_loss So Far: 0.6913703083992004
Total elapsed time: 00h 04m 10s
Epoch 1/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 23s 219ms/step - accuracy: 0.4743 - loss: 0.6938 - val_accuracy: 0.3438 - val_loss: 0.6944
Epoch 2/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 22s 219ms/step - accuracy: 0.3486 - loss: 0.6935 - val_accuracy: 0.3438 - val_loss: 0.6933
Epoch 3/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 22s 218ms/step - accuracy: 0.6385 - loss: 0.6932 - val_accuracy: 0.6562 - val_loss: 0.6929
Epoch 4/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 22s 221ms/step - accuracy: 0.4223 - loss: 0.6932 - val_accuracy: 0.3438 - val_loss: 0.6935
Epoch 5/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 22s 216ms/step - accuracy: 0.4507 - loss: 0.6932 - val_accuracy: 0.3470 - val_loss: 0.6932


In [11]:
best_model = automodel.export_model()
best_model.summary()

c:\Users\Asus\Downloads\Uni_work\Grad-Project\autokeras_env\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cast_to_float32 (CastToFloat32) │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 158, 158, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 156, 156, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 78, 78, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 76, 76, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 74, 74, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 37, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification_head_1           │ (None, 1)              │             0 │
│ (Activation)                    │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,137 (184.13 KB)

 Trainable params: 47,137 (184.13 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
raw_scores = best_model.predict(test_ds)
raw_scores[:5]

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step


array([[0.49781308],
       [0.4966996 ],
       [0.4975272 ],
       [0.49677098],
       [0.49770075]], dtype=float32)

In [15]:
if raw_scores.ndim == 1 or raw_scores.shape[-1] == 1:
    y_score = raw_scores.reshape(-1)
    y_pred = (y_score >= 0.5).astype(int)
else:
    y_score = raw_scores[:, 1]
    y_pred = raw_scores.argmax(axis=1)

y_pred[:10], y_score[:10]

(array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([0.49781308, 0.4966996 , 0.4975272 , 0.49677098, 0.49770075,
        0.4973368 , 0.49695462, 0.4973219 , 0.49664134, 0.4966468 ],
       dtype=float32))

In [16]:
y_test = test_df["class_label"].map({"NonStroke": 0, "Stroke": 1}).to_numpy()

metrics = evaluate_binary_classifier(
    y_true=y_test,
    y_pred=y_pred,
    y_score=y_score,
)

metrics

{'accuracy': 0.6540880503144654,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.47139423076923076}

In [17]:
target_names = [INVERSE_LABEL_MAPPING[0], INVERSE_LABEL_MAPPING[1]]

report = create_classification_report_dict(
    y_true=y_test,
    y_pred=y_pred,
    target_names=target_names,
)

report

{'NonStroke': {'precision': 0.6540880503144654,
  'recall': 1.0,
  'f1-score': 0.7908745247148289,
  'support': 208.0},
 'Stroke': {'precision': 0.0,
  'recall': 0.0,
  'f1-score': 0.0,
  'support': 110.0},
 'accuracy': 0.6540880503144654,
 'macro avg': {'precision': 0.3270440251572327,
  'recall': 0.5,
  'f1-score': 0.39543726235741444,
  'support': 318.0},
 'weighted avg': {'precision': 0.4278311775641786,
  'recall': 0.6540880503144654,
  'f1-score': 0.5173015759141019,
  'support': 318.0}}

In [18]:
save_confusion_matrix_plot(
    y_true=y_test,
    y_pred=y_pred,
    class_names=target_names,
    output_path=FACE_AUTOKERAS_TRIAL_001_CONFUSION_MATRIX_PATH,
)

FACE_AUTOKERAS_TRIAL_001_CONFUSION_MATRIX_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/face_trial_001_autokeras_vanilla_160_confusion_matrix.png')

In [19]:
FACE_AUTOKERAS_TRIAL_001_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df = test_df.copy()
predictions_df["true_label_encoded"] = y_test
predictions_df["predicted_label_encoded"] = y_pred
predictions_df["stroke_probability"] = y_score
predictions_df["true_label"] = predictions_df["true_label_encoded"].map(INVERSE_LABEL_MAPPING)
predictions_df["predicted_label"] = predictions_df["predicted_label_encoded"].map(INVERSE_LABEL_MAPPING)
predictions_df["is_correct"] = predictions_df["true_label_encoded"] == predictions_df["predicted_label_encoded"]

predictions_df.to_csv(FACE_AUTOKERAS_TRIAL_001_PREDICTIONS_PATH, index=False)

FACE_AUTOKERAS_TRIAL_001_PREDICTIONS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/face_trial_001_autokeras_vanilla_160_predictions.csv')

In [20]:
save_markdown_results(
    metrics=metrics,
    report=report,
    output_path=FACE_AUTOKERAS_TRIAL_001_RESULTS_PATH,
    title="Face Trial 001 AutoKeras Vanilla Baseline Results",
)

FACE_AUTOKERAS_TRIAL_001_RESULTS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/experiments/face_trial_001_autokeras_vanilla_160_results.md')

In [21]:
best_model.save(FACE_AUTOKERAS_TRIAL_001_MODEL_PATH)

FACE_AUTOKERAS_TRIAL_001_MODEL_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/models/experiments/face/trial_001_autokeras_vanilla_160/model_unusable.keras')

## Experiment Summary: Trial 001 (AutoKeras Vanilla Baseline)

### 1. Objective & Setup
* **Goal:** Establish an initial AutoML pipeline baseline using a highly constrained search space to validate the end-to-end data streaming (`tf.data`) and evaluation infrastructure.
* **Configuration:** `block_type="vanilla"` (shallow custom CNN search space), `max_trials=2`, `epochs=5`, image resolution $160 \times 160$. Class imbalance handled via mathematical loss weighting (`class_weight`).

### 2. Key Findings & Architecture
* **Pipeline Integrity:** **Success.** The data pipeline smoothly handled batch loading, on-the-fly augmentation, and model evaluation without encountering Out-of-Memory (OOM) errors or kernel crashes.
* **Selected Model:** AutoKeras converged on a minimal 4-layer custom CNN utilizing `GlobalAveragePooling2D` with **47,137 trainable parameters (~184 KB)**. 

### 3. Performance Analysis (Underfitting)
* **Binary Cross-Entropy Loss:** Flatlined at $\approx 0.693$, indicating the optimizer failed to break past a random-guess gradient plateau. 
* **Class Imbalance Vulnerability:** Due to the severe lack of capacity in a shallow 4-layer architecture over only 5 epochs, the network failed to learn complex facial mesh features. Instead, it minimized loss by defaulting to predicting the majority class (`NonStroke`) for 100% of samples.
* **Metrics Summary:** * Total Accuracy: **65.4%** (Identical to the majority-class distribution threshold).
  * Minority Class (`Stroke`) Recall: **0.00** ### 4. Next Engineering Steps
This trial served exclusively as a **Pipeline Validation Experiment** rather than a viable performance benchmark. 

To break out of the underfitting plateau while respecting the deployment's edge/offline hardware constraints, **Trial 002** will pivot to establishing a **True Defensible Baseline** utilizing a frozen, pre-trained lightweight backbone (**MobileNetV2**) to leverage robust, pre-existing facial feature extraction capabilities.